# Работа 7. Сбор данных с lifehacker.ru

Задача — собрать заголовки и тексты материалов рубрики
[Технологии](https://lifehacker.ru/topics/technology/) с первых десяти страниц,
используя `requests` и `BeautifulSoup`.

Порядок работы: определяем формат пагинации → находим в разметке классы блоков со
ссылками → собираем ссылки с десяти страниц → скачиваем каждый материал →
вытаскиваем заголовок и текст → складываем в датафрейм.

## 1. Формат ссылки для пагинации

Кнопки перелистывания на странице рубрики лежат в блоке `lh-the-paginator`, а
внутри — ссылки вида `/topics/technology/?page=2`. То есть номер страницы
передаётся параметром `page`, а не сегментом пути (`/page/2/` отдаёт 404).

In [1]:
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://lifehacker.ru"
TOPIC_URL = f"{BASE_URL}/topics/technology/"
PAGES = 10
DELAY = 0.4          # пауза между запросами, чтобы не создавать нагрузку на сайт

# Сайт отдаёт контент только «браузерам», поэтому подставляем User-Agent.
# Session переиспользует TCP-соединение — 310 запросов проходят заметно быстрее.
session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"),
    "Accept-Language": "ru-RU,ru;q=0.9",
})


def fetch(url: str, attempts: int = 3) -> str | None:
    """Скачивает страницу, повторяя попытку при сетевой ошибке.

    Возвращает None, если страница так и не открылась: одна недоступная статья
    не должна ронять сбор остальных трёхсот.
    """
    for attempt in range(1, attempts + 1):
        try:
            response = session.get(url, timeout=30)
            if response.status_code == 200:
                return response.text
            print(f"    {url} -> статус {response.status_code}")
            return None
        except requests.RequestException as error:
            print(f"    попытка {attempt}/{attempts} для {url}: {type(error).__name__}")
            time.sleep(2 * attempt)
    return None


print("страница пагинации:", f"{TOPIC_URL}?page=2")

страница пагинации: https://lifehacker.ru/topics/technology/?page=2


## 2. Классы блоков с материалами

В разметке списка каждая карточка материала — это `div.lh-small-article-card`,
внутри неё ссылка `a.lh-small-article-card__link`. Заголовок в самой ссылке
отсутствует (он в соседнем блоке `__title`), но продублирован в атрибуте `title`,
поэтому ссылку и предварительный заголовок можно взять из одного элемента.

На странице материала заголовок лежит в `h1`, а текст — в
`article.single-article__post-content`.

In [2]:
CARD_LINK = "a.lh-small-article-card__link[href]"
ARTICLE_BODY = "article.single-article__post-content"


def extract_links(page_html: str) -> list[str]:
    """Достаёт со страницы списка абсолютные ссылки на материалы."""
    soup = BeautifulSoup(page_html, "lxml")
    links = []
    for tag in soup.select(CARD_LINK):
        href = tag["href"]
        # отсекаем ссылки на рубрики и внешние домены — нужны только материалы
        if href.startswith("/") and not href.startswith("/topics/"):
            links.append(urljoin(BASE_URL, href))
    return links


# Проверяем на первой странице, что селектор рабочий
first_page = fetch(TOPIC_URL)
sample_links = extract_links(first_page)
print(f"ссылок на первой странице: {len(sample_links)}")
sample_links[:3]

ссылок на первой странице: 30


['https://lifehacker.ru/sravnenie-iphone-18-pro-i-17-pro/',
 'https://lifehacker.ru/ii-mozhet-unichtozhit-chelovechestvo-v-blizhaishee-desyatiletie/',
 'https://lifehacker.ru/programmy-dlya-formatirovaniya-fleshki/']

## 3. Ссылки с десяти страниц списка

In [3]:
article_links: list[str] = []
seen: set[str] = set()

for page in range(1, PAGES + 1):
    url = TOPIC_URL if page == 1 else f"{TOPIC_URL}?page={page}"
    html = fetch(url)
    if html is None:
        print(f"страница {page}: не получена, пропускаем")
        continue

    found = extract_links(html)
    # материалы могут повторяться между страницами, если список обновился
    # прямо во время сбора, поэтому храним множество уже виденных ссылок
    new = [link for link in found if link not in seen]
    seen.update(new)
    article_links.extend(new)

    print(f"страница {page:>2}: найдено {len(found):>3}, новых {len(new):>3}, "
          f"всего {len(article_links):>3}")
    time.sleep(DELAY)

print(f"\nвсего уникальных ссылок: {len(article_links)}")

страница  1: найдено  30, новых  30, всего  30
страница  2: найдено  30, новых  30, всего  60
страница  3: найдено  30, новых  30, всего  90
страница  4: найдено  30, новых  30, всего 120
страница  5: найдено  30, новых  30, всего 150
страница  6: найдено  30, новых  30, всего 180
страница  7: найдено  30, новых  30, всего 210
страница  8: найдено  30, новых  30, всего 240
страница  9: найдено  30, новых  30, всего 270
страница 10: найдено  30, новых  30, всего 300

всего уникальных ссылок: 300


## 4. Скачивание и разбор каждого материала

In [4]:
def parse_article(html: str) -> tuple[str, str]:
    """Возвращает заголовок и текст материала.

    Текст собираем не целиком из контейнера, а по абзацам и подзаголовкам:
    так в результат не попадают подписи к картинкам, кнопки и блоки рекламы,
    которые лежат в том же контейнере отдельными элементами.
    """
    soup = BeautifulSoup(html, "lxml")

    heading = soup.find("h1")
    title = heading.get_text(" ", strip=True) if heading else ""

    body = soup.select_one(ARTICLE_BODY)
    if body is None:
        return title, ""

    parts = [tag.get_text(" ", strip=True) for tag in body.find_all(["p", "h2", "h3"])]
    text = "\n".join(part for part in parts if part)
    return title, text


records = []
failed = []

for number, link in enumerate(article_links, start=1):
    html = fetch(link)
    if html is None:
        failed.append(link)
        continue

    title, text = parse_article(html)
    records.append({"url": link, "title": title, "text": text,
                    "text_length": len(text)})

    if number % 25 == 0 or number == len(article_links):
        print(f"обработано {number:>3} из {len(article_links)}")
    time.sleep(DELAY)

print(f"\nсобрано материалов: {len(records)}, не удалось: {len(failed)}")

обработано  25 из 300
обработано  50 из 300
обработано  75 из 300
обработано 100 из 300
обработано 125 из 300
обработано 150 из 300
обработано 175 из 300
обработано 200 из 300
обработано 225 из 300
обработано 250 из 300
обработано 275 из 300
обработано 300 из 300

собрано материалов: 300, не удалось: 0


## 5. Датафрейм с результатами

In [5]:
articles = pd.DataFrame(records)
print(f"строк: {articles.shape[0]}, столбцов: {articles.shape[1]}")
articles.head()

строк: 300, столбцов: 4


,url,title,text,text_length
0,https://lifehacker.ru/sravnenie-iphone-18-pro-...,Чем новые iPhone 18 Pro и Pro Max отличаются о...,"9 сентября Apple провела презентацию, на котор...",8380
1,https://lifehacker.ru/ii-mozhet-unichtozhit-ch...,ИИ может уничтожить человечество в ближайшее д...,Исследователь по безопасности ИИ Эван Хьюбинге...,3330
2,https://lifehacker.ru/programmy-dlya-formatiro...,9 лучших бесплатных программ для форматировани...,Флешки и карты памяти со временем неизбежно сб...,5354
3,https://lifehacker.ru/iphone-duo-otzyvy/,В Сети неоднозначно приняли iPhone Duo — многи...,"Вчера, 9 сентября, Apple представила свой перв...",3777
4,https://lifehacker.ru/iphone-18-pro-cena-v-ros...,МТС открыла предзаказ на iPhone 18 Pro и склад...,МТС первой среди российских ретейлеров объявил...,700


In [6]:
# Проверяем качество сбора: пустые заголовки и тексты означают,
# что селектор не сработал на каком-то нестандартном материале
print(f"пустых заголовков: {(articles['title'] == '').sum()}")
print(f"пустых текстов:    {(articles['text'] == '').sum()}")
print(f"дубликатов по url: {articles['url'].duplicated().sum()}")
print()
print(articles["text_length"].describe().round(0).to_string())

пустых заголовков: 0
пустых текстов:    0
дубликатов по url: 0

count      300.0
mean      3802.0
std       3650.0
min        646.0
25%       1419.0
50%       1970.0
75%       5555.0
max      20008.0


In [7]:
# Пример собранного материала целиком
example = articles.loc[articles["text_length"].idxmax()]
print("ЗАГОЛОВОК:", example["title"])
print("URL:      ", example["url"])
print("ДЛИНА:    ", example["text_length"], "символов")
print("\nНАЧАЛО ТЕКСТА:")
print(example["text"][:700])

ЗАГОЛОВОК: 12 лучших нейросетей для генерации изображений с нуля в 2026 году
URL:       https://lifehacker.ru/neiroseti-dlya-generacii-izobrazhenii/
ДЛИНА:     20008 символов

НАЧАЛО ТЕКСТА:
Сейчас нейросетью для генерации изображений уже никого не удивишь: ИИ-картинки повсюду, их можно делать практически в каждом чат-боте, и для многих даже не нужно уметь составлять сложные промпты. Но какой конкретный сервис выбрать — вопрос сложный. Чтобы упростить вам жизнь, изучили имеющиеся на рынке варианты и выделили лучшие — по качеству генераций, удобству использования и модификации результата, а также просто по доступности.
🥰 Опрос для дорогих читателей! Помогите нам стать лучше и поделитесь , что вы хотели бы изменить. В конце — бонус от наших иллюстраторов.
Для наглядности все нейросети протестировали на двух одинаковых запросах:
В сервисах, где за раз генерируется несколько вариант


In [8]:
# Сохраняем результат: сбор зависит от живого сайта, и без файла
# повторный запуск ноутбука даст уже другой набор материалов
articles.to_csv("lifehacker_technology.csv", index=False)
print("сохранено в lifehacker_technology.csv")
print(f"размер файла: {round(len(articles.to_csv(index=False).encode()) / 1024)} КБ")

сохранено в lifehacker_technology.csv
размер файла: 2052 КБ


## Итог

Собраны заголовки и тексты материалов с первых десяти страниц рубрики
«Технологии». Пагинация задаётся параметром `?page=N`, на каждой странице по 30
карточек, ссылки берутся из `a.lh-small-article-card__link`, а на странице
материала заголовок читается из `h1` и текст — из абзацев внутри
`article.single-article__post-content`.

Два момента, которые стоит держать в голове:

**Повторный запуск даст другой результат.** Рубрика пополняется, и вчерашняя
первая страница завтра станет второй. Поэтому собранное сохранено в
`lifehacker_technology.csv` — именно этот файл фиксирует результат работы.

**В текст попадают редакционные врезки.** Внутри статей встречаются вставки вроде
приглашения пройти опрос: разметкой они не отличаются от обычных абзацев, поэтому
отделить их можно только по содержанию, а не по классам.